In [ ]:
from PIL import Image
import requests
from io import BytesIO
import sys
sys.path.append('../')
import sam3
from sam3.train.data.collator import collate_fn_api as collate
from sam3.model.utils.misc import copy_data_to_device
import os
sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")

In [ ]:
import torch
# turn on tfloat32 for Ampere GPUs
# https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook. If your card doesn't support it, try float16 instead
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

# inference mode for the whole notebook. Disable if you need gradients
torch.inference_mode().__enter__()

In [ ]:
import sys
import numpy as np
import matplotlib

# Ép sử dụng backend 'Agg' (chuyên dùng để render ảnh tĩnh, không phụ thuộc vào Jupyter)
# HOẶC bạn có thể thử 'TkAgg' nếu muốn biểu đồ bật lên thành một cửa sổ riêng.
matplotlib.use('Agg') 

import matplotlib.pyplot as plt

if sam3_root not in sys.path:
    sys.path.append(sam3_root)
sys.path.append(f"{sam3_root}/examples")

from sam3.visualization_utils import plot_results, show_mask, show_box, show_points


In [ ]:
from umpt_sam.umpa_model import UMPAModel
from umpt_sam.config.model_config import UMPAModelConfig, UPFEConfig, MPPGConfig

device = "cuda" if torch.cuda.is_available() else "cpu"

model_config = UMPAModelConfig(
    checkpoint_path=os.path.join(sam3_root, "model_trained/CVC_300_DICE_93,45_MIOU_87,81_eps_18.pth"),
    image_size=1008,
    embed_dim=256,
    text_embed_dim=512,
    freeze_image_encoder=True,
    upfe=UPFEConfig(scoring_hidden_dim=256),
    mppg=MPPGConfig(),
)

model = UMPAModel.from_config(model_config=model_config).to(device)
model.eval()


In [ ]:
image_path = os.path.join(sam3_root, "data", "data_benmarks/CVC-300/images/160.png")
image_pil = Image.open(image_path).convert("RGB")

image_size = model_config.image_size
image_resized = image_pil.resize((image_size, image_size))
image_np = np.array(image_resized)

image_tensor = torch.from_numpy(image_np).permute(2, 0, 1).float()
image_tensor = image_tensor / 127.5 - 1.0
image_tensor = image_tensor.unsqueeze(0).to(device)

# Example prompts in pixel coordinates (update to match your image)
H, W = image_size, image_size
# box = torch.tensor([[[0.25 * W, 0.25 * H, 0.75 * W, 0.75 * H]]], dtype=torch.float32, device=device)
points = torch.tensor([[[0.5 * W, 0.5 * H], [0.65 * W, 0.65 * H]]], dtype=torch.float32, device=device)
point_labels = torch.tensor([[1, 0]], dtype=torch.int64, device=device)
captions = ["lesion"]


In [ ]:
with torch.inference_mode():
    points = torch.tensor([[[0.5 * W, 0.5 * H]]], dtype=torch.float32, device=device)
    point_labels = torch.tensor([[1]], dtype=torch.int64, device=device)

    outputs = model(
        image=image_tensor,
        points=points,
        point_labels=point_labels,
        captions=["lesion"],  # optional
        multimask_output=False,
    )

pred_masks = outputs["pred_masks"]
if pred_masks.shape[-2:] != (image_size, image_size):
    pred_masks = torch.nn.functional.interpolate(
        pred_masks, size=(image_size, image_size), mode="bilinear", align_corners=False
    )
mask_prob = torch.sigmoid(pred_masks[0, 0]).cpu().numpy()
mask_bin = mask_prob > 0.5

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image_resized)
show_mask(mask_bin, ax)
# show_box(box[0, 0].detach().cpu().numpy(), ax)
show_points(points[0].detach().cpu().numpy(), point_labels[0].detach().cpu().numpy(), ax)
ax.axis("off")
plt.savefig("output.png", dpi=300, bbox_inches='tight')
plt.show()